# Environmental Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the hurricane-track walkthrough in [1.6-geospatial-vector-data-environmental-exercises.ipynb](1.6-geospatial-vector-data-environmental-exercises.ipynb).
:::

## Exercise 1: Hurricane track analysis


*Can you quickly find out which US states Hurricane Florence passed through using geopandas?*

Apply geopandas to read in the geospatial data, plot, and analyse the track of Hurricane Florence
from 30 August to 18 September 2018. The track is the archived advisory record from the
[National Hurricane Center](https://www.nhc.noaa.gov/); the state boundaries are a
[US Census Bureau](https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html)
cartographic boundary file.
References:

1. [Introduction to GeoPandas](https://geopandas.org/en/stable/getting_started/introduction.html) — geopandas' official website
2. [Geopandas: an introduction](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-2/geopandas-an-introduction.html) — Automating GIS Processes
3. [Use Data for Earth and Environmental Science in Open Source Python](https://www.earthdatascience.org/courses/use-data-open-source-python/)
4. [The Shapely User Manual](https://shapely.readthedocs.io/en/stable/manual.html)
5. [Geospatial Analysis with Python and R](https://kodu.ut.ee/~kmoch/geopython2020/index.html)

In [ ]:
# Pre-supplied: download and cache the two real data files.
import pooch

states_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/gz_2010_us_040_00_5m.json",
    known_hash="sha256:7a8c022e063a34a83f35984cde6c81992ece5983f8cc4459ed02e40687739573",
    fname="us_states.geojson",
    path=pooch.os_cache("mlees"),
)
track_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/florence.csv",
    known_hash="sha256:385691583ee41682a1c905e042c56ea362609fb04645934bdda7911c55c8b63f",
    fname="florence.csv",
    path=pooch.os_cache("mlees"),
)

**Q1) Import geopandas and pandas.**

Import geopandas as `gpd`, pandas as `pd`, and matplotlib's pyplot as `plt`.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

**Q2) Read the state boundary file with geopandas' `read_file` function.**

The file is cached at `states_path`. Call the result `country`.

In [ ]:
country = gpd.read_file(states_path)
print(country.shape)

**Q3) Have a look at the data. What type of geometries does it contain?**

Print the first few rows, and answer the geometry question in a comment.

In [ ]:
print(country.head())
print(country.geom_type.unique())

# Both Polygon and MultiPolygon: a state with offshore islands cannot be one ring, so the
# file mixes the two. Anything that assumes pure Polygon will break on this data.

**Q4) Have a look at the data on a map using geopandas' `.plot()` method.**

Exclude Alaska and Hawaii using the `NAME` attribute and pandas' `.isin()` method. Specify the
figsize to be 30 x 20.

In [ ]:
lower48 = country[country["NAME"].isin(["Alaska", "Hawaii"]) == False]
lower48.plot(figsize=(30, 20))
plt.show()

In [ ]:
# Pre-supplied: read in the hurricane Florence data, drop the advisory bookkeeping columns,
# fix the longitude sign, and have a look at the dataframe.
florence = pd.read_csv(track_path)
florence = florence.drop(["AdvisoryNumber", "Forecaster", "Received"], axis=1)
florence["Long"] = 0 - florence["Long"]
florence.head(3)

**Q5) Create a GeoDataFrame from the `florence` DataFrame.**

Build the geometry column from the `Long` and `Lat` columns with
[`gpd.points_from_xy`](https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html),
and call the result `gdf_florence`. Then look at its first few rows.

In [ ]:
gdf_florence = gpd.GeoDataFrame(
    florence,
    geometry=gpd.points_from_xy(florence["Long"], florence["Lat"]),
    crs="EPSG:4326",
)
print(gdf_florence.head(2))
print(gdf_florence.geom_type.unique())

**Q6) Plot the US states map (without Alaska and Hawaii) and hurricane Florence together.**

Draw the states as a base map, then plot the hurricane positions on top in a colour that stands
out.

In [ ]:
fig, ax = plt.subplots(1, figsize=(30, 20))
base = lower48.plot(ax=ax, color="#3B3C6E")
gdf_florence.plot(ax=base, color="darkred", marker="*", markersize=10)
plt.show()

**Q7) What is the coordinate reference system of the data?**

Check it for both layers, and say in a comment whether they match — two layers only line up on
one pair of axes if they share a crs.

In [ ]:
print("states:", country.crs)
print("track: ", gdf_florence.crs)

# Both are WGS84 geographic coordinates (EPSG:4326), so the two layers line up as plotted.
# Had they differed, one would have to be reprojected with .to_crs before overlaying.

**Q8) Which states did the hurricane pass through?**

```{hint}
One approach is to plot the trajectory over the US map, annotate each state with its name, and
read the answer off the figure. Other approaches are more than welcome — `overlay` with
`how="intersection"` will give you the track points that fall inside a state polygon, and the
`NAME` column of the result then answers the question directly.
```

In [ ]:
fig, ax = plt.subplots(figsize=(30, 20))
lower48.plot(ax=ax)

# annotate each state at its own centroid
for name, centre in zip(lower48["NAME"], lower48.centroid):
    ax.annotate(name, xy=(centre.x, centre.y), ha="center")

# the overlay keeps only the track points that fall inside a state polygon
res_intersection = gdf_florence.overlay(country, how="intersection")
res_intersection.plot(ax=ax, color="red", marker="*", markersize=25)
plt.show()

print(res_intersection["NAME"].unique())